# Recommendation Systems using Collaborative Filtering
## Scenario 1: User-Based CF
## Scenario 2: Item-Based CF
Includes: RMSE, MAE, Precision@K, Visualizations

In [ ]:
!wget http://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt

## Load Dataset

In [ ]:
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id','movie_id','rating','timestamp'])
movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1', names=['movie_id','title'] + [f'col{i}' for i in range(22)])
movies = movies[['movie_id','title']]
data = pd.merge(ratings, movies, on='movie_id')
data.head()

## User-Based CF

In [ ]:
user_item = data.pivot_table(index='user_id', columns='title', values='rating')
user_item_filled = user_item.fillna(0)
user_sim = cosine_similarity(user_item_filled)
user_sim_df = pd.DataFrame(user_sim, index=user_item.index, columns=user_item.index)

In [ ]:
def predict_user(user, movie):
    sims = user_sim_df[user].sort_values(ascending=False)[1:6]
    num, den = 0,0
    for u,s in sims.items():
        r = user_item.loc[u, movie]
        if not np.isnan(r):
            num += s*r
            den += s
    return num/den if den!=0 else 0

In [ ]:
def recommend_user(user, n=5):
    unseen = user_item.loc[user][user_item.loc[user].isna()].index
    scores = {m:predict_user(user,m) for m in unseen}
    return sorted(scores.items(), key=lambda x:x[1], reverse=True)[:n]
recommend_user(1)

## Item-Based CF

In [ ]:
item_user = data.pivot_table(index='title', columns='user_id', values='rating').fillna(0)
item_sim = cosine_similarity(item_user)
item_sim_df = pd.DataFrame(item_sim, index=item_user.index, columns=item_user.index)

In [ ]:
def recommend_item(user, n=5):
    user_r = user_item.loc[user].dropna()
    scores = {}
    for m,r in user_r.items():
        sims = item_sim_df[m].sort_values(ascending=False)[1:6]
        for sm,s in sims.items():
            if sm not in user_r:
                scores[sm] = scores.get(sm,0)+s*r
    return sorted(scores.items(), key=lambda x:x[1], reverse=True)[:n]
recommend_item(1)

## Evaluation

In [ ]:
actual, pred = [], []
for u in user_item.index[:50]:
    for m in user_item.columns[:50]:
        if not np.isnan(user_item.loc[u,m]):
            actual.append(user_item.loc[u,m])
            pred.append(predict_user(u,m))
rmse = sqrt(mean_squared_error(actual,pred))
mae = mean_absolute_error(actual,pred)
print('RMSE:',rmse)
print('MAE:',mae)

In [ ]:
def precision_at_k(user, k=5):
    recs = [m for m,_ in recommend_item(user,k)]
    actual = user_item.loc[user].dropna().index
    hits = len(set(recs)&set(actual))
    return hits/k
print('Precision@5:', precision_at_k(1))

## Visualization

In [ ]:
plt.figure()
sns.heatmap(user_item_filled.iloc[:20,:20])
plt.title('User-Item Matrix')
plt.show()

In [ ]:
plt.figure()
sns.heatmap(user_sim_df.iloc[:20,:20])
plt.title('User Similarity')
plt.show()